# Observer Design Pattern 

explained using the YouTube Channel example.

#### The Concept

The Observer pattern defines a one-to-many dependency. When one object (The Subject) changes state, all its dependents (Observers) are notified and updated automatically.

#### Analogy:

A Magazine Subscription.
- **Subject (Publisher)**: Doesn't know who exactly subscribes. It just has a list of addresses. When a new issue is ready, it sends it to everyone on the list.
- **Observer (Subscriber)**: You subscribe once. You get updates automatically until you unsubscribe. You don't need to call the publisher every day asking, "Is the new issue out?"

## The Classic OOP Way (Java-Style)

In the strict OOP approach, we use Interfaces for both the Subject and the Observer to decouple them. The Subject maintains a list of objects that implement the `Observer` interface.

#### THE OBSERVER INTERFACE

In [1]:
from abc import ABC, abstractmethod

class Observer(ABC):
    @abstractmethod
    def update(self, channel_name: str, video_title: str):
        pass

#### THE SUBJECT (Publisher)

In [2]:
from abc import ABC
from typing import List

class Subject(ABC):
    def __init__(self):
        self._observers: List[Observer] = []

    def attach(self, observer: Observer):
        print("Subject: Attached an observer.")
        self._observers.append(observer)

    def detach(self, observer: Observer):
        self._observers.remove(observer)

    def notify(self, channel_name: str, video_title: str):
        print("Subject: Notifying observers...")
        for observer in self._observers:
            observer.update(channel_name, video_title)

#### CONCRETE SUBJECT (YouTube Channel)

In [3]:
class YouTubeChannel(Subject):
    def __init__(self, name: str):
        super().__init__()
        self.name = name
        self.latest_video = ""

    def upload_video(self, title: str):
        print(f"\n[{self.name}] Uploading video: '{title}'")
        self.latest_video = title
        # Trigger the notification
        self.notify(self.name, title)

#### CONCRETE OBSERVERS (Subscribers)

In [4]:
class Subscriber(Observer):
    def __init__(self, username: str):
        self.username = username

    def update(self, channel_name: str, video_title: str):
        # Implementation of the interface method
        print(f"   -> Hey {self.username}, {channel_name} just uploaded '{video_title}'!")

#### CLIENT CODE

In [5]:
def main():
    channel = YouTubeChannel("TechDaily")
    
    sub1 = Subscriber("Alice")
    sub2 = Subscriber("Bob")

    channel.attach(sub1)
    channel.attach(sub2)

    channel.upload_video("Python Observer Pattern Tutorial")
    
    channel.detach(sub1) # Alice unsubscribes
    
    channel.upload_video("Java vs Python") # Only Bob gets this

if __name__ == "__main__":
    main()

Subject: Attached an observer.
Subject: Attached an observer.

[TechDaily] Uploading video: 'Python Observer Pattern Tutorial'
Subject: Notifying observers...
   -> Hey Alice, TechDaily just uploaded 'Python Observer Pattern Tutorial'!
   -> Hey Bob, TechDaily just uploaded 'Python Observer Pattern Tutorial'!

[TechDaily] Uploading video: 'Java vs Python'
Subject: Notifying observers...
   -> Hey Bob, TechDaily just uploaded 'Java vs Python'!


## The Pythonic Way

In Python, functions are first-class citizens. We don't need to force subscribers to be a class inheriting from `Observer`. They can be simple functions, lambdas, or class methods. We can simply store a list of Callables (functions).

#### THE PYTHONIC SUBJECT

In [6]:
from typing import List, Callable

class YouTubeChannel:
    def __init__(self, name: str):
        self.name = name
        # The list holds functions, not necessarily objects
        self.subscribers: List[Callable[[str, str], None]] = []

    def subscribe(self, callback_func):
        self.subscribers.append(callback_func)

    def unsubscribe(self, callback_func):
        self.subscribers.remove(callback_func)

    def upload(self, title: str):
        print(f"\n[{self.name}] Uploading: '{title}'")
        # Just loop and call the functions!
        for callback in self.subscribers:
            callback(self.name, title)

#### THE OBSERVERS (Can be anything callable)

In [7]:
# A simple function
def push_notification(channel, title):
    print(f"   📲 PUSH: New video on {channel}: {title}")

# A class method
class EmailService:
    def send_email(self, channel, title):
        print(f"   📧 EMAIL: Dear user, check out {title} on {channel}")

#### CLIENT CODE

In [8]:
def main():
    channel = YouTubeChannel("CodeWithMe")
    
    email_service = EmailService()

    # 1. Subscribe a function
    channel.subscribe(push_notification)
    
    # 2. Subscribe a method
    channel.subscribe(email_service.send_email)
    
    # 3. Subscribe a Lambda (Anonymous function)
    channel.subscribe(lambda c, t: print(f"   📝 LOG: {c} uploaded {t}"))

    # Trigger
    channel.upload("Mastering Python Decorators")

    # Unsubscribe
    channel.unsubscribe(push_notification)
    
    channel.upload("Django Crash Course")

if __name__ == "__main__":
    main()


[CodeWithMe] Uploading: 'Mastering Python Decorators'
   📲 PUSH: New video on CodeWithMe: Mastering Python Decorators
   📧 EMAIL: Dear user, check out Mastering Python Decorators on CodeWithMe
   📝 LOG: CodeWithMe uploaded Mastering Python Decorators

[CodeWithMe] Uploading: 'Django Crash Course'
   📧 EMAIL: Dear user, check out Django Crash Course on CodeWithMe
   📝 LOG: CodeWithMe uploaded Django Crash Course


#### Key Differences

| Feature          | Classic OOP                                                        | Pythonic                                                         |
|------------------|--------------------------------------------------------------------|------------------------------------------------------------------|
| **Observer Type** | Must be a class implementing an `Observer` interface.             | Can be a function, method, lambda, or class.                    |
| **Storage**       | List of objects (`List[Observer]`).                                | List of callables (`List[Callable]`).                           |
| **Coupling**      | Tighter coupling — subscriber depends on observer class structure.| Zero coupling — subscriber only needs matching call signature.  |


#### When to use which?

- **Java Way**: When you have complex observers that need to maintain state (e.g., a GUI window that needs to remember its previous color before updating).
- **Python Way**: For 95% of event handling, callbacks, and simple notifications. It's how libraries like Tkinter (GUI) or Django Signals work.

# Observer Design Pattern 

explained with a complex, real-world example: A Stock Market Trading System.

#### The Scenario: Real-Time Stock Ticker

The Stock Market is volatile. Prices change every millisecond. Different parts of a trading system need to react differently to these changes:
- **Mobile App**: Needs to update the UI (User Interface) so the user sees the new price.
- **Algorithm Bot**: Needs to analyze the price immediately to decide whether to Buy or Sell.
- **Logger**: Needs to record the price change in a database for audit history.

It is inefficient for the App, Bot, and Logger to constantly query the Stock Exchange ("Is the price new? Is the price new?"). Instead, the Stock Exchange should Notify them when a change happens.

## The Classic OOP Way (Java-Style)

We define a strict `Observer` interface. Every component (App, Bot, Logger) must implement `update()`. The Subject (`StockMarket`) keeps a list of these observers.

#### THE OBSERVER INTERFACE

In [10]:
from abc import ABC, abstractmethod

class StockObserver(ABC):
    @abstractmethod
    def update(self, ticker: str, price: float):
        pass

#### THE SUBJECT (Stock Exchange)

In [11]:
from typing import List, Dict

class StockExchange:
    def __init__(self):
        # We Map Ticker Symbol -> List of Observers
        # e.g., "AAPL" -> [MobileApp, Logger]
        self._observers: Dict[str, List[StockObserver]] = {}
        self._prices: Dict[str, float] = {}

    def register(self, ticker: str, observer: StockObserver):
        if ticker not in self._observers:
            self._observers[ticker] = []
        self._observers[ticker].append(observer)
        print(f"[System] Registered {observer.__class__.__name__} for {ticker}")

    def unregister(self, ticker: str, observer: StockObserver):
        if ticker in self._observers:
            self._observers[ticker].remove(observer)

    def set_price(self, ticker: str, price: float):
        print(f"\n--- Market Update: {ticker} is now ${price} ---")
        self._prices[ticker] = price
        self._notify(ticker, price)

    def _notify(self, ticker: str, price: float):
        if ticker in self._observers:
            for observer in self._observers[ticker]:
                observer.update(ticker, price)

#### CONCRETE OBSERVERS

In [12]:
class MobileApp(StockObserver):
    def update(self, ticker: str, price: float):
        print(f"📱 Mobile App: Updating UI. {ticker} -> ${price}")

class TradingBot(StockObserver):
    def update(self, ticker: str, price: float):
        if price < 150:
            print(f"🤖 Algo Bot: BUY signal for {ticker} at ${price}!")
        else:
            print(f"🤖 Algo Bot: Holding {ticker}...")

class AuditLogger(StockObserver):
    def update(self, ticker: str, price: float):
        print(f"💾 Logger: Saved {ticker}:${price} to database.")

#### CLIENT CODE

In [13]:
def main():
    market = StockExchange()

    app = MobileApp()
    bot = TradingBot()
    logger = AuditLogger()

    # Registering for specific channels
    market.register("AAPL", app)
    market.register("AAPL", bot)
    market.register("GOOGL", logger)
    market.register("GOOGL", app)

    # Updates
    market.set_price("AAPL", 180.0) # App & Bot notified
    market.set_price("AAPL", 145.0) # App & Bot notified (Bot triggers BUY)
    market.set_price("GOOGL", 2800.0) # Logger & App notified

if __name__ == "__main__":
    main()

[System] Registered MobileApp for AAPL
[System] Registered TradingBot for AAPL
[System] Registered AuditLogger for GOOGL
[System] Registered MobileApp for GOOGL

--- Market Update: AAPL is now $180.0 ---
📱 Mobile App: Updating UI. AAPL -> $180.0
🤖 Algo Bot: Holding AAPL...

--- Market Update: AAPL is now $145.0 ---
📱 Mobile App: Updating UI. AAPL -> $145.0
🤖 Algo Bot: BUY signal for AAPL at $145.0!

--- Market Update: GOOGL is now $2800.0 ---
💾 Logger: Saved GOOGL:$2800.0 to database.
📱 Mobile App: Updating UI. GOOGL -> $2800.0


## The Pythonic Way (Events / Callbacks)

In Python, we can simplify this using a **Pub/Sub (Publisher-Subscriber)** approach with dictionaries and functions. We don't need an `Observer` interface. We just need to know "Which function do I call when AAPL changes?".

We can also use **Decorators** to register observers elegantly.

#### THE PYTHONIC SUBJECT (Event Manager)

In [14]:
from typing import Callable

class StockMarket:
    def __init__(self):
        # Dictionary: Ticker -> List of Functions
        self._subscribers = defaultdict(list)

    def subscribe(self, ticker: str):
        """
        A Decorator to register functions easily!
        @market.subscribe("AAPL")
        def my_function...
        """
        def decorator(func: Callable):
            self._subscribers[ticker].append(func)
            print(f"[System] Subscribed '{func.__name__}' to {ticker}")
            return func
        return decorator

    def update_price(self, ticker: str, price: float):
        print(f"\n--- 📈 Market Update: {ticker} ${price} ---")
        if ticker in self._subscribers:
            for callback in self._subscribers[ticker]:
                # Directly calling the function
                callback(ticker, price)

#### CLIENT CODE (Using Decorators)

In [17]:
from collections import defaultdict

def main():
    market = StockMarket()

    # --- Defining Observers Inline ---
    
    @market.subscribe("TSLA")
    def mobile_ui_updater(ticker, price):
        print(f"📱 UI: Updating Tesla chart to {price}")

    @market.subscribe("TSLA")
    def elon_tracker(ticker, price):
        if price < 500:
            print("🚨 Alert: TSLA is crashing! Tweet something!")

    @market.subscribe("BTC")
    def crypto_wallet(ticker, price):
        print(f"💰 Wallet: Bitcoin value is now {price}")

    # --- Triggering Events ---
    
    market.update_price("TSLA", 750.00)
    market.update_price("BTC", 45000.00)
    market.update_price("TSLA", 400.00) # Triggers Alert

    # Notice: No classes needed for observers. 
    # Logic is kept near the subscription point.

if __name__ == "__main__":
    main()

[System] Subscribed 'mobile_ui_updater' to TSLA
[System] Subscribed 'elon_tracker' to TSLA
[System] Subscribed 'crypto_wallet' to BTC

--- 📈 Market Update: TSLA $750.0 ---
📱 UI: Updating Tesla chart to 750.0

--- 📈 Market Update: BTC $45000.0 ---
💰 Wallet: Bitcoin value is now 45000.0

--- 📈 Market Update: TSLA $400.0 ---
📱 UI: Updating Tesla chart to 400.0
🚨 Alert: TSLA is crashing! Tweet something!


#### Why the Pythonic version is better here

- **Decorators** (`@subscribe`): This makes the code extremely readable. You can see exactly what triggers a function right above its definition.
- **Decoupling**: The `mobile_ui_updater` function doesn't need to know about a `StockObserver` class hierarchy. It just needs to accept `ticker` and `price` arguments.
- **Flexibility**: You can mix functions, class methods, and lambdas in the same subscriber list.

#### Real-World Python usage

This pattern is heavily used in:
- **Django Signals**: `@receiver(post_save)` triggers actions when a database row is saved.
- **Flask**: `@app.route("/")` observes the URL and triggers a function when that URL is requested.
- **FastAPI / Celery**: Using decorators to bind events to handlers.